# Dispersão da umidade relativa: SONDA-MD × NASA POWER

Compara os horários coincidentes e válidos das duas bases tratadas. Os 176 horários sem umidade na SONDA-MD são excluídos apenas desta comparação, sem preenchimento.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr

def localizar_dados_tratados():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidato = base / '2026' / 'dados_tratados'
        if ((candidato / 'sonda_md_horario_padronizado.csv').exists()
                and (candidato / 'nasa_power_horario_padronizado.csv').exists()):
            return candidato.resolve()
    raise FileNotFoundError('Não foi encontrada a pasta 2026/dados_tratados.')

PASTA_DADOS = localizar_dados_tratados()
sonda = pd.read_csv(PASTA_DADOS / 'sonda_md_horario_padronizado.csv', parse_dates=['data_hora'])
nasa = pd.read_csv(PASTA_DADOS / 'nasa_power_horario_padronizado.csv', parse_dates=['data_hora'])

dados = (
    sonda[['data_hora', 'umidade_relativa']]
    .merge(
        nasa[['data_hora', 'umidade_relativa']],
        on='data_hora',
        how='inner',
        suffixes=('_sonda_md', '_nasa_power'),
    )
    .dropna(subset=['umidade_relativa_sonda_md', 'umidade_relativa_nasa_power'])
    .sort_values('data_hora')
    .reset_index(drop=True)
)

if len(dados) != 17_392:
    raise ValueError(f'Esperados 17.392 pares válidos; encontrados {len(dados):,}.')

r, p_valor = pearsonr(dados['umidade_relativa_sonda_md'], dados['umidade_relativa_nasa_power'])
n_formatado = f'{len(dados):,}'.replace(',', '.')
r_formatado = f'{r:.3f}'.replace('.', ',')
print(f'Pares horários coincidentes e válidos: {n_formatado}')
print(f'Período: {dados.data_hora.min()} a {dados.data_hora.max()}')
print(f'Correlação de Pearson: r = {r_formatado}')

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7), dpi=150)
ax.scatter(
    dados['umidade_relativa_sonda_md'],
    dados['umidade_relativa_nasa_power'],
    s=9, alpha=0.28, color='#2166ac', edgecolors='none', rasterized=True,
)

limite_min = min(dados['umidade_relativa_sonda_md'].min(), dados['umidade_relativa_nasa_power'].min())
limite_max = max(dados['umidade_relativa_sonda_md'].max(), dados['umidade_relativa_nasa_power'].max())
margem = 0.03 * (limite_max - limite_min)
limites = [limite_min - margem, limite_max + margem]
ax.plot(limites, limites, color='#333333', linewidth=1.4, label='y = x')
ax.set_xlim(limites)
ax.set_ylim(limites)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Umidade relativa da SONDA-MD (%)')
ax.set_ylabel('Umidade relativa da NASA POWER (%)')
ax.set_title('Umidade relativa horária: SONDA-MD × NASA POWER')
ax.text(
    0.04, 0.96, f'n = {n_formatado}\nPearson: r = {r_formatado}',
    transform=ax.transAxes, ha='left', va='top', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.9, edgecolor='0.7'),
)
ax.grid(True, linestyle=':', alpha=0.35)
ax.legend(loc='lower right')
fig.tight_layout()

PASTA_SAIDA = PASTA_DADOS.parent / 'resultados' / 'figuras' / 'comparacao_umidade'
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_FIGURA = PASTA_SAIDA / 'dispersao_umidade_sonda_md_nasa_power.png'
ARQUIVO_TABELA = PASTA_SAIDA / 'pares_umidade_sonda_md_nasa_power.csv'
fig.savefig(ARQUIVO_FIGURA, dpi=300, bbox_inches='tight')
dados.to_csv(ARQUIVO_TABELA, index=False)
print(f'Figura salva em: {ARQUIVO_FIGURA}')
print(f'Tabela salva em: {ARQUIVO_TABELA}')
plt.show()